# 🚗 ATMS-Net Phase 1 — Vehicle Detector Training on Kaggle GPU

**Adaptive Traffic Management System with Emergency Preemption**

This notebook trains the custom YOLO-style vehicle detector from scratch on the
MS COCO 2017 vehicle subset using Kaggle's free Tesla T4 GPU.

### What This Notebook Does
1. Clones the ATMS-Net codebase from GitHub
2. Installs all dependencies
3. Downloads & prepares the COCO vehicle dataset (car, motorcycle, bus, truck)
4. Trains the custom 13.2M parameter detector for 50 epochs
5. Saves the best checkpoint based on mAP@0.5

### Requirements
- **GPU**: Enable GPU in Settings → Accelerator → GPU T4 x2
- **Internet**: Enable Internet in Settings → Internet → On
- **Estimated Time**: ~2-3 hours for 50 epochs

---

## 1. Environment Setup

In [ ]:
# ============================================================
# 1.1 — Verify GPU is available
# ============================================================
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:      {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
    DEVICE = 'cuda'
else:
    print("⚠ No GPU detected! Enable GPU in Settings → Accelerator")
    print("  Training on CPU will be extremely slow.")
    DEVICE = 'cpu'

In [ ]:
# ============================================================
# 1.2 — Clone the ATMS-Net repository
# ============================================================
import os

REPO_URL = 'https://github.com/CodeNebula-Dev/Adaptive-Traffic-Management-System-with-Emergency-Preemption.git'
PROJECT_DIR = '/kaggle/working/ATMS-Net'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print(f"\n✓ Repository cloned to {PROJECT_DIR}")
else:
    print(f"✓ Repository already exists at {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ============================================================
# 1.3 — Install dependencies
# ============================================================
!pip install -q pycocotools opencv-python-headless tqdm matplotlib seaborn PyYAML Pillow
print("\n✓ All dependencies installed")

In [ ]:
# ============================================================
# 1.4 — Verify project structure
# ============================================================
import sys
sys.path.insert(0, PROJECT_DIR)

# Quick sanity check — try importing the model
from models.detector.yolo_detector import ATMSDetector

model = ATMSDetector(num_classes=4)
model.summary()
del model
print("\n✓ Model imports successfully")

---
## 2. Dataset Preparation

We download the MS COCO 2017 dataset and filter it to only the 4 vehicle classes:
- **car** (COCO id=3) → class 0
- **motorcycle** (COCO id=4) → class 1
- **bus** (COCO id=6) → class 2
- **truck** (COCO id=8) → class 3

Labels are converted from COCO format to YOLO format (normalized `cx cy w h`).

In [ ]:
# ============================================================
# 2.1 — Download COCO 2017 dataset
# ============================================================
# This downloads ~20GB of data. Takes 5-15 minutes on Kaggle.
#
# If you already have COCO on Kaggle as a dataset, you can skip
# the download and just set the paths manually (see cell 2.1b).
# ============================================================

DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'coco')

# Option A: Download COCO from scratch
!python data/coco/download_coco.py --data-dir {DATA_DIR} --download

In [ ]:
# ============================================================
# 2.1b — (ALTERNATIVE) Use Kaggle's pre-loaded COCO dataset
# ============================================================
# If you added "COCO 2017" as a Kaggle dataset input, uncomment
# the lines below and comment out cell 2.1 above.
#
# This avoids re-downloading and saves 10-15 minutes.
# ============================================================

# import shutil
#
# KAGGLE_COCO = '/kaggle/input/coco-2017-dataset/coco2017'
#
# # Link images and annotations to our expected directory structure
# os.makedirs(DATA_DIR, exist_ok=True)
#
# if os.path.exists(KAGGLE_COCO):
#     # Create symlinks to avoid copying 20GB of data
#     for item in ['train2017', 'val2017', 'annotations']:
#         src = os.path.join(KAGGLE_COCO, item)
#         dst = os.path.join(DATA_DIR, item)
#         if os.path.exists(src) and not os.path.exists(dst):
#             os.symlink(src, dst)
#             print(f"  ✓ Linked {item}")
#
#     # Run the label extraction (no download)
#     !python data/coco/download_coco.py --data-dir {DATA_DIR}
#     print("\n✓ COCO dataset linked and labels extracted")
# else:
#     print("⚠ Kaggle COCO dataset not found at expected path.")
#     print("  Add it: + Add Data → Search 'COCO 2017' → Add")

In [ ]:
# ============================================================
# 2.2 — Verify dataset is ready
# ============================================================
train_txt = os.path.join(DATA_DIR, 'train.txt')
val_txt = os.path.join(DATA_DIR, 'val.txt')

assert os.path.exists(train_txt), f"Missing: {train_txt}"
assert os.path.exists(val_txt), f"Missing: {val_txt}"

with open(train_txt) as f:
    n_train = len(f.readlines())
with open(val_txt) as f:
    n_val = len(f.readlines())

print(f"✓ Training images:   {n_train:,}")
print(f"✓ Validation images: {n_val:,}")
print(f"✓ Total:             {n_train + n_val:,}")

In [ ]:
# ============================================================
# 2.3 — Visualize sample training images with labels
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLASS_NAMES = ['car', 'motorcycle', 'bus', 'truck']
COLORS = [(0.2, 0.7, 0.3), (0.9, 0.3, 0.1), (0.1, 0.5, 0.9), (0.8, 0.6, 0.1)]

with open(train_txt) as f:
    all_images = [line.strip() for line in f.readlines()]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('COCO Vehicle Subset — Sample Training Images', fontsize=16, fontweight='bold')

np.random.seed(42)
sample_indices = np.random.choice(len(all_images), 6, replace=False)

for idx, ax in zip(sample_indices, axes.flat):
    img_path = all_images[idx]
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax.imshow(img)

    # Load corresponding label
    label_path = img_path.replace('train2017', 'labels/train2017')
    label_path = label_path.replace('.jpg', '.txt').replace('.png', '.txt')

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:]]

                # Convert from normalized YOLO to pixel coords
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                box_w = bw * w
                box_h = bh * h

                rect = Rectangle((x1, y1), box_w, box_h,
                                 linewidth=2, edgecolor=COLORS[cls_id],
                                 facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-5, CLASS_NAMES[cls_id],
                        color='white', fontsize=10, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2',
                                  facecolor=COLORS[cls_id], alpha=0.8))

    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_training_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved sample_training_images.png")

---
## 3. Training Configuration

We modify the default config to optimize for Kaggle's Tesla T4 GPU.

In [ ]:
# ============================================================
# 3.1 — Create Kaggle-optimized training config
# ============================================================
import yaml

# Load the default config
with open('configs/detector.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ---- Kaggle-specific overrides ----

# GPU-optimized batch size (T4 has 16GB VRAM)
config['training']['batch_size'] = 32

# Training epochs
config['training']['epochs'] = 50

# Enable mixed precision (FP16) — 2x memory savings on T4
config['training']['mixed_precision'] = True

# More dataloader workers on Kaggle (4 CPU cores)
config['data']['num_workers'] = 4

# Update data paths for Kaggle
config['data']['data_dir'] = DATA_DIR
config['data']['train_list'] = train_txt
config['data']['val_list'] = val_txt
config['data']['label_dir'] = os.path.join(DATA_DIR, 'labels', 'train2017')

# Save checkpoints in working directory (persists after kernel restart)
config['checkpoint']['save_dir'] = '/kaggle/working/checkpoints'
config['logging']['log_dir'] = '/kaggle/working/logs'

# Save the modified config
kaggle_config_path = 'configs/kaggle_detector.yaml'
with open(kaggle_config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✓ Kaggle training config:")
print(f"  Epochs:          {config['training']['epochs']}")
print(f"  Batch size:      {config['training']['batch_size']}")
print(f"  Mixed precision: {config['training']['mixed_precision']}")
print(f"  Image size:      {config['model']['img_size']}")
print(f"  Learning rate:   {config['training']['learning_rate']}")
print(f"  Checkpoints:     {config['checkpoint']['save_dir']}")

---
## 4. Training

This is the main training loop. It will:
1. Build the model (13.2M parameters)
2. Train for 50 epochs with warmup + cosine annealing LR
3. Validate every epoch and compute mAP@0.5
4. Save the best model checkpoint

**Expected time: ~2-3 hours on T4 GPU**

In [ ]:
# ============================================================
# 4.1 — Run the full training
# ============================================================
!python scripts/train_detector.py \
    --config configs/kaggle_detector.yaml \
    --device cuda

---
## 5. Results & Evaluation

In [ ]:
# ============================================================
# 5.1 — Plot training loss curves
# ============================================================
import matplotlib.pyplot as plt
import re

log_file = '/kaggle/working/logs/training.log'

if os.path.exists(log_file):
    epochs_log = []
    losses = {'total': [], 'box': [], 'obj': [], 'cls': []}
    lrs = []

    with open(log_file) as f:
        for line in f:
            parts = dict(item.split('=') for item in line.strip().split())
            epochs_log.append(int(parts['epoch']))
            losses['total'].append(float(parts['loss']))
            losses['box'].append(float(parts['box']))
            losses['obj'].append(float(parts['obj']))
            losses['cls'].append(float(parts['cls']))
            lrs.append(float(parts['lr']))

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle('ATMS-Net Phase 1 — Training Progress', fontsize=16, fontweight='bold')

    # Total loss
    axes[0].plot(epochs_log, losses['total'], 'b-', linewidth=2, label='Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Total Loss')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Component losses
    axes[1].plot(epochs_log, losses['box'], 'r-', linewidth=1.5, label='Box (CIoU)')
    axes[1].plot(epochs_log, losses['obj'], 'g-', linewidth=1.5, label='Objectness')
    axes[1].plot(epochs_log, losses['cls'], 'm-', linewidth=1.5, label='Classification')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Loss Components')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # Learning rate schedule
    axes[2].plot(epochs_log, lrs, 'orange', linewidth=2)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Learning Rate Schedule')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved training_curves.png")
else:
    print("⚠ No training log found. Run training first (Section 4).")

In [ ]:
# ============================================================
# 5.2 — Load and inspect the best checkpoint
# ============================================================
import torch

ckpt_path = '/kaggle/working/checkpoints/best.pt'

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')

    print("=" * 50)
    print("Best Checkpoint Summary")
    print("=" * 50)
    print(f"  Saved at epoch:  {ckpt['epoch'] + 1}")
    print(f"  Best mAP@0.5:    {ckpt['best_map']:.4f}")
    print(f"  Model tensors:   {len(ckpt['model_state_dict'])}")
    if 'ema_state_dict' in ckpt:
        print(f"  EMA tensors:     {len(ckpt['ema_state_dict'])}")
    print(f"  File size:       {os.path.getsize(ckpt_path) / 1024**2:.1f} MB")
    print("=" * 50)
else:
    print("⚠ No best.pt checkpoint found. Run training first (Section 4).")
    # Check for last.pt fallback
    last_path = '/kaggle/working/checkpoints/last.pt'
    if os.path.exists(last_path):
        print(f"  Found last.pt ({os.path.getsize(last_path)/1024**2:.1f} MB)")

In [ ]:
# ============================================================
# 5.3 — Run inference on sample validation images
# ============================================================
import sys
sys.path.insert(0, PROJECT_DIR)

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from models.detector.yolo_detector import ATMSDetector
from utils.nms import batch_nms

CLASS_NAMES = ['car', 'motorcycle', 'bus', 'truck']
COLORS = [(0.2, 0.7, 0.3), (0.9, 0.3, 0.1), (0.1, 0.5, 0.9), (0.8, 0.6, 0.1)]

ckpt_path = '/kaggle/working/checkpoints/best.pt'

if os.path.exists(ckpt_path):
    # Load model
    ckpt = torch.load(ckpt_path, map_location='cuda')
    model = ATMSDetector(num_classes=4)
    
    # Try loading EMA weights first (smoother predictions)
    if 'ema_state_dict' in ckpt:
        model.load_state_dict(ckpt['ema_state_dict'])
        print("Loaded EMA model weights")
    else:
        model.load_state_dict(ckpt['model_state_dict'])
        print("Loaded model weights")
    
    model = model.to('cuda')
    model.eval()

    # Load sample validation images
    with open(val_txt) as f:
        val_images = [line.strip() for line in f.readlines()]

    np.random.seed(123)
    sample_imgs = np.random.choice(val_images, 6, replace=False)

    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    fig.suptitle('ATMS-Net — Inference Results on Validation Images',
                 fontsize=16, fontweight='bold')

    for img_path, ax in zip(sample_imgs, axes.flat):
        img_orig = cv2.imread(img_path)
        if img_orig is None:
            continue
        img_orig = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)
        h_orig, w_orig = img_orig.shape[:2]

        # Preprocess: resize to 416x416, normalize to [0,1]
        img_resized = cv2.resize(img_orig, (416, 416))
        img_tensor = torch.from_numpy(img_resized).float() / 255.0
        img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0).to('cuda')

        # Inference
        with torch.no_grad():
            predictions = model(img_tensor)

        # NMS
        detections = batch_nms(
            predictions,
            conf_threshold=0.25,
            iou_threshold=0.45,
            max_detections=100,
        )

        # Draw on original image
        ax.imshow(img_orig)

        if detections[0] is not None and len(detections[0]) > 0:
            dets = detections[0].cpu().numpy()
            for det in dets:
                # Scale from 416x416 back to original size
                cx, cy, w, h_box = det[0], det[1], det[2], det[3]
                conf = det[4]
                cls_scores = det[5:]
                cls_id = int(np.argmax(cls_scores))
                score = conf * cls_scores[cls_id]

                if score < 0.3:
                    continue

                # Convert to pixel coords on original image
                x1 = (cx - w/2) / 416 * w_orig
                y1 = (cy - h_box/2) / 416 * h_orig
                box_w = w / 416 * w_orig
                box_h = h_box / 416 * h_orig

                rect = Rectangle((x1, y1), box_w, box_h,
                                 linewidth=2, edgecolor=COLORS[cls_id],
                                 facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-5, f"{CLASS_NAMES[cls_id]} {score:.2f}",
                        color='white', fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2',
                                  facecolor=COLORS[cls_id], alpha=0.85))

        ax.axis('off')

    plt.tight_layout()
    plt.savefig('/kaggle/working/inference_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved inference_results.png")
else:
    print("⚠ No checkpoint found. Run training first (Section 4).")

---
## 6. Export & Download

Download the trained model checkpoint to use locally in Phase 2.

In [ ]:
# ============================================================
# 6.1 — Package outputs for download
# ============================================================
import shutil

output_dir = '/kaggle/working/atms_net_trained'
os.makedirs(output_dir, exist_ok=True)

# Copy checkpoint
for pt_file in ['best.pt', 'last.pt']:
    src = f'/kaggle/working/checkpoints/{pt_file}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(output_dir, pt_file))
        print(f"  ✓ Copied {pt_file} ({os.path.getsize(src)/1024**2:.1f} MB)")

# Copy training log
log_src = '/kaggle/working/logs/training.log'
if os.path.exists(log_src):
    shutil.copy2(log_src, os.path.join(output_dir, 'training.log'))
    print("  ✓ Copied training.log")

# Copy plots
for img_file in ['training_curves.png', 'inference_results.png', 'sample_training_images.png']:
    src = f'/kaggle/working/{img_file}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(output_dir, img_file))
        print(f"  ✓ Copied {img_file}")

# Create zip for easy download
shutil.make_archive('/kaggle/working/atms_net_phase1_trained', 'zip', output_dir)
zip_size = os.path.getsize('/kaggle/working/atms_net_phase1_trained.zip') / 1024**2
print(f"\n✓ Created atms_net_phase1_trained.zip ({zip_size:.1f} MB)")
print("\n  Download from: Output → atms_net_phase1_trained.zip")
print("  Place best.pt in your local project's checkpoints/ folder")

In [ ]:
# ============================================================
# 6.2 — Final summary
# ============================================================
print("=" * 60)
print("ATMS-Net Phase 1 — Training Complete!")
print("=" * 60)

ckpt_path = '/kaggle/working/checkpoints/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print(f"  Best mAP@0.5:    {ckpt['best_map']:.4f}")
    print(f"  Trained epochs:  {ckpt['epoch'] + 1}")
    print(f"  Model size:      {os.path.getsize(ckpt_path)/1024**2:.1f} MB")

print(f"\n  Next steps:")
print(f"  1. Download best.pt from the Output tab")
print(f"  2. Place it in your local checkpoints/ folder")
print(f"  3. Move to Phase 2: Traffic Signal Controller (DQN)")
print("=" * 60)